<a href="https://colab.research.google.com/github/1BF24CS033/AI/blob/main/1BF24CS033_Week_6_Unifuing_Logic_Expressions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import re

def is_variable(x):
    return isinstance(x, str) and x[0].isupper()

def is_constant(x):
    return isinstance(x, str) and x[0].islower() and '(' not in x

def parse_expression(expr):
    expr = expr.replace(' ', '')
    match = re.match(r'^([a-zA-Z0-9_]+)\((.*)\)$', expr)
    if not match:
        return expr

    op = match.group(1)
    args_str = match.group(2)

    args = []
    bracket_count = 0
    current_arg = []
    for char in args_str:
        if char == ',' and bracket_count == 0:
            args.append(''.join(current_arg))
            current_arg = []
        else:
            if char == '(':
                bracket_count += 1
            elif char == ')':
                bracket_count -= 1
            current_arg.append(char)
    if current_arg:
        args.append(''.join(current_arg))

    parsed_args = [parse_expression(arg) for arg in args]
    return [op] + parsed_args

def occurs_check(var, expr):
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_check(var, arg) for arg in expr[1:])
    return False

def apply_substitution(subst, expr):
    if not subst:
        return expr
    if is_variable(expr):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [expr[0]] + [apply_substitution(subst, arg) for arg in expr[1:]]
    return expr

def unify(x, y):
    if x == y:
        return {}

    if is_variable(x):
        if occurs_check(x, y):
            return "FAILURE"
        else:
            return {x: y}

    if is_variable(y):
        if occurs_check(y, x):
            return "FAILURE"
        else:
            return {y: x}

    if isinstance(x, list) and isinstance(y, list):
        if x[0] != y[0] or len(x) != len(y):
            return "FAILURE"

        subst = {}
        for i in range(1, len(x)):
            arg1 = apply_substitution(subst, x[i])
            arg2 = apply_substitution(subst, y[i])

            s = unify(arg1, arg2)
            if s == "FAILURE":
                return "FAILURE"

            for key in subst:
                subst[key] = apply_substitution(s, subst[key])
            for key, val in s.items():
                if key not in subst:
                    subst[key] = val
        return subst

    return "FAILURE"

In [11]:
def run_unification():
    print("--- Logic Unification Program ---")
    print("Example format: p(a, X) or p(Y, b)\n")

    expr1_str = input("Enter the first expression (default: p(a, X)): ").strip() or "p(a, X)"
    expr2_str = input("Enter the second expression (default: p(Y, b)): ").strip() or "p(Y, b)"

    try:
        expr1 = parse_expression(expr1_str)
        expr2 = parse_expression(expr2_str)

        print(f"\nParsed Expression 1: {expr1}")
        print(f"Parsed Expression 2: {expr2}")

        result = unify(expr1, expr2)

        if result == "FAILURE":
            print("\nResult: Unification Failed!")
        else:
            print("\nResult (Most General Unifier):")
            if not result:
                print("No substitution needed (already identical).")
            else:
                def to_str(e):
                    return f"{e[0]}({', '.join(to_str(arg) for arg in e[1:])})" if isinstance(e, list) else str(e)

                for var, val in result.items():
                    print(f"  {var} / {to_str(val)}")

                # Apply the final substitutions to the original expression to show the unified output
                unified_expr = apply_substitution(result, expr1)
                print(f"\nUnified Expression: {to_str(unified_expr)}")
    except Exception as e:
        print(f"An error occurred during parsing or execution: {e}")

run_unification()

--- Logic Unification Program ---
Example format: p(a, X) or p(Y, b)

Enter the first expression (default: p(a, X)): p(a, b, Z)
Enter the second expression (default: p(Y, b)): p(X, Y)

Parsed Expression 1: ['p', 'a', 'b', 'Z']
Parsed Expression 2: ['p', 'X', 'Y']

Result: Unification Failed!
